# data-collection_005

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (52).ipynb`

| Field | Value |
|---|---|
| Section | `data_collection` |
| Market | `us_equities` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 31 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas_datareader as pdr
import yfinance as yf
import datetime

In [ ]:
# Define the date range
start = datetime.datetime(2019, 1, 1)
end = datetime.datetime(2024, 1, 1)

# Fetch the data
df = yf.download('TATAMOTORS.NS', start=start, end=end)

# Display the first few rows of the dataframe
df.head()

In [ ]:
df.tail()

In [ ]:
print(df.columns)

In [ ]:
df = df.reset_index()
df.head()

In [ ]:
df = df.drop(['Adj Close'], axis=1)
df.head()

In [ ]:
plt.plot(df.Close)

In [ ]:
df

In [ ]:
ma21 = df.Close.rolling(21).mean()
ma21

In [ ]:
ma10 = df.Close.rolling(10).mean()
ma10

In [ ]:
plt.figure(figsize = (12,6))
plt.plot(df.Close)
plt.plot(ma21, 'g')
plt.plot(ma10, 'y')

In [ ]:
df.shape

In [ ]:
df['10-day'] = df['Close'].rolling(10).mean()
df['21-day'] = df['Close'].rolling(21).mean()
df

In [ ]:
df['signal'] = np.where(df['10-day'] > df['21-day'], 1, 0)
df['signal'] = np.where(df['10-day'] < df['21-day'], -1, df['signal'])
df.dropna(inplace=True)
df.head()

In [ ]:
df['return'] = np.log(df['Close']).diff()
df['signal-return'] = df['return'] * df['signal']
df.tail()

In [ ]:
df = df.drop(['High', 'Low', 'Volume'], axis = 1)
df

In [ ]:
df['entry'] = df.signal.diff()
df

In [ ]:
plt.figure(figsize = (12, 6))
plt.plot(df.Close)
plt.plot(ma21, 'r')
plt.plot(ma10, 'y')
plt.plot(df['10-day'][df.entry == 2], '^', color = 'g', markersize = 5)
plt.plot(df['21-day'][df.entry == -2], '^', color = 'r', markersize = 7)

In [ ]:
plt.plot(np.exp(df['return']).cumprod(), label = 'Buy/Hold')
plt.plot(np.exp(df['signal-return']).cumprod(), label = 'signal')

In [ ]:
df['buy'] = (df['10-day'][df.entry == 2]).astype(int)
df['sell'] = (df['10-day'][df.entry == -2]).astype(int)
df[1:50]

In [ ]:
df['trade'] = df['buy'].fillna(df['sell']).combine_first(df['sell'])
df

In [ ]:
df.trade.count()

In [ ]:
# Drop NaN values but keep the index for later
valid_numbers = df['trade'].dropna()

# Calculate the difference between consecutive numbers
differences = valid_numbers.diff()

# Create a new column with NaN where the original column had NaN, and differences where numbers existed
df['difference'] = differences.reindex(df.index)


In [ ]:
df[100:150]

In [ ]:
df.difference.unique().astype(int)

In [ ]:
df.difference.sum()

In [ ]:
df.difference.sum()/df.Close.mean() * 100

In [ ]:
df.difference.max()

In [ ]:
df.difference.min()